## its working like able to recommend music after detection but need adjustment for detection ( accuracy is bad) , add ai companion , music is recommended but not poping up la


In [1]:
!python -m pip install --upgrade pip



In [2]:
pip --version

pip 24.3.1 from c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\pip (python 3.12)

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip uninstall sklearn


Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install opencv-python dlib scipy pyttsx3 pandas speechrecognition


Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install pyaudio


Note: you may need to restart the kernel to use updated packages.


In [7]:
import tkinter as tk
from tkinter import messagebox
import winsound
import threading
import cv2
import dlib
import numpy as np
from scipy.spatial import distance as dist
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import pyttsx3
import speech_recognition as sr



In [8]:
# Initialize dlib’s face detector and landmark predictor
face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Initialize text-to-speech engine
engine = pyttsx3.init()
engine.setProperty('rate', 150)

In [9]:
# Thresholds for drowsiness detection
EAR_THRESHOLD = 0.3
CONSEC_FRAMES_THRESHOLD = 20
drowsy_frame_count = 0

In [10]:
# Function to calculate Eye Aspect Ratio (EAR)
def calculate_EAR(eye):
    A = dist.euclidean(eye[1], eye[5])
    B = dist.euclidean(eye[2], eye[4])
    C = dist.euclidean(eye[0], eye[3])
    ear = (A + B) / (2.0 * C)
    return ear

In [11]:
# Music recommendation function
def recommend_music():
    data = pd.read_excel('data1.xlsx')
    data = data.drop_duplicates(subset=['name'])

    # Normalize and cluster
    col_features = ['danceability', 'energy', 'valence', 'loudness']
    X = MinMaxScaler().fit_transform(data[col_features])
    kmeans = KMeans(init="k-means++", n_clusters=2, random_state=15).fit(X)
    data['kmeans'] = kmeans.labels_

    # Recommend a random song from the "high energy" cluster (kmeans label 1)
    recommended_song = data[data['kmeans'] == 1]['name'].sample(1).values[0]
    return recommended_song

In [12]:
# Alarm function
def play_alarm():
    winsound.Beep(1000, 1000)  # Frequency 1000 Hz, Duration 1000 ms


In [13]:
# Function to display popup and handle user response
def show_popup(song):
    def on_yes():
        popup.destroy()
        engine.say(f"Playing {song}")
        engine.runAndWait()

    def on_no():
        popup.destroy()

    def timeout():
        if not user_responded[0]:
            play_alarm()
            popup.destroy()

    user_responded = [False]

    popup = tk.Tk()
    popup.title("Music Recommendation")
    label = tk.Label(popup, text=f"Would you like to play '{song}'?", font=("Helvetica", 14))
    label.pack(pady=10)

    yes_button = tk.Button(popup, text="Yes", command=lambda: [on_yes(), user_responded.__setitem__(0, True)])
    yes_button.pack(side="left", padx=20, pady=10)

    no_button = tk.Button(popup, text="No", command=lambda: [on_no(), user_responded.__setitem__(0, True)])
    no_button.pack(side="right", padx=20, pady=10)

    popup.after(10000, timeout)  # Close popup after 10 seconds if no response
    popup.mainloop()

In [14]:
# Initialize webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector(gray)

    for face in faces:
        landmarks = landmark_predictor(gray, face)
        landmarks = np.array([[p.x, p.y] for p in landmarks.parts()])

        left_eye = landmarks[36:42]
        right_eye = landmarks[42:48]
        left_ear = calculate_EAR(left_eye)
        right_ear = calculate_EAR(right_eye)
        ear = (left_ear + right_ear) / 2.0

        if ear < EAR_THRESHOLD:
            drowsy_frame_count += 1
        else:
            drowsy_frame_count = 0

        if drowsy_frame_count >= CONSEC_FRAMES_THRESHOLD:
            cv2.putText(frame, "Drowsy Detected!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            engine.say("Drowsiness detected. Recommending music.")
            engine.runAndWait()

            song = recommend_music()
            threading.Thread(target=show_popup, args=(song,)).start()

    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


FileNotFoundError: [Errno 2] No such file or directory: 'data1.xlsx'

In [ ]:
'''import speech_recognition as sr

recognizer = sr.Recognizer()
with sr.Microphone() as source:
    print("Speak something...")
    audio = recognizer.listen(source)
    print("Audio captured successfully!")'''


Speak something...
Audio captured successfully!


In [ ]:
'''print(sr.Microphone.list_microphone_names())''''


['Microsoft Sound Mapper - Input', 'Microphone (Realtek(R) Audio)', 'Microsoft Sound Mapper - Output', 'Speakers (Realtek(R) Audio)', 'Primary Sound Capture Driver', 'Microphone (Realtek(R) Audio)', 'Primary Sound Driver', 'Speakers (Realtek(R) Audio)', 'Speakers (Realtek(R) Audio)', 'Microphone (Realtek(R) Audio)', 'Microphone (Realtek HD Audio Mic input)', 'Speakers 1 (Realtek HD Audio output with HAP)', 'Speakers 2 (Realtek HD Audio output with HAP)', 'PC Speaker (Realtek HD Audio output with HAP)', 'Microphone (Realtek Digital Microphone)', 'Headphones (Realtek HD Audio 2nd output)', 'Stereo Mix (Realtek HD Audio Stereo input)', 'Headset (@System32\\drivers\\bthhfenum.sys,#2;%1 Hands-Free%0\r\n;(Baseus Bowie WM01))', 'Headset (@System32\\drivers\\bthhfenum.sys,#2;%1 Hands-Free%0\r\n;(Baseus Bowie WM01))', 'Headphones ()']


In [ ]:
'''# Main loop for drowsiness detection

true_labels = [...]  # Ground truth labels (0: awake, 1: drowsy) for test data
predicted_labels = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector(gray)

    for face in faces:
        landmarks = landmark_predictor(gray, face)
        landmarks = np.array([[p.x, p.y] for p in landmarks.parts()])

        left_eye = landmarks[36:42]
        right_eye = landmarks[42:48]
        left_ear = calculate_EAR(left_eye)
        right_ear = calculate_EAR(right_eye)
        ear = (left_ear + right_ear) / 2.0

        if ear < EAR_THRESHOLD:
            drowsy_frame_count += 1
        else:
            drowsy_frame_count = 0

        if drowsy_frame_count >= CONSEC_FRAMES_THRESHOLD:
            cv2.putText(frame, "Drowsy Detected!", (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            engine.say("Drowsiness detected. Would you like to play some music?") #untill here its working
            engine.runAndWait()

            #listn for voice response
            with sr.Microphone() as source:
                try:
                    print("Adjusting for ambient noise...")
                    recognizer.adjust_for_ambient_noise(source, duration=1)
                    print("Listening for your response...Say 'yes' to play music.")
                    audio = recognizer.listen(source, timeout=10)
                    print("Processing your response...")
                    command = recognizer.recognize_google(audio)
                    if "yes" in command.lower():
                        print("Detected 'yes', Recommending music...")
                        recommend_music()
                    else:
                         print("No valid response detected.")
                        
                except sr.WaitTimeoutError:
                    print("Timeout reached. No response detected.")
                except sr.UnknownValueError:
                    print("Could not understand the audio.")
                except sr.RequestError as e:
                    print(f"Could not request results; {e}")

    cv2.imshow("Drowsiness Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()'''

Adjusting for ambient noise...
Listening for your response...Say 'yes' to play music.
Processing your response...
Could not understand the audio.
Adjusting for ambient noise...
Listening for your response...Say 'yes' to play music.
Processing your response...
Could not understand the audio.
Adjusting for ambient noise...
Listening for your response...Say 'yes' to play music.


In [ ]:
'''from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Ensure `true_labels` and `predicted_labels` are of the same length
accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels)
f1 = f1_score(true_labels, predicted_labels)
cm = confusion_matrix(true_labels, predicted_labels)

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print("Confusion Matrix:")
print(cm)'''
